# Dev Assistant · finding evidence in Order & Inventory

Does semantic search find code that keyword search misses? We compare both methods on the same immutable snapshot, then inspect the answers Qwen produced with vector retrieval. The two missing-evidence questions are scored separately. The dataset was labelled before running this comparison. Open the saved outputs to inspect the run without installing a model.

In [1]:
import json
import platform
from dataclasses import dataclass
from pathlib import Path
from time import perf_counter
from typing import NotRequired, TypedDict

from dotenv import load_dotenv
from sqlalchemy import text

from dev_assistant.corpus import Passage
from dev_assistant.model import Answer, Ollama
from dev_assistant.service import retrieve
from dev_assistant.storage import current, database

class Question(TypedDict):
    id: str
    split: str
    question: str
    path: str | None
    lines: NotRequired[list[int]]


class Dataset(TypedDict):
    repository: str
    commit: str
    questions: list[Question]


root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
load_dotenv(root / ".env")
dataset: Dataset = json.loads((root / "notebooks/data/questions.json").read_text())
db = database(root / ".env")
model = Ollama()
with db.connect() as conn:
    state = current(conn)
    assert (state.url, state.revision) == (dataset["repository"], dataset["commit"])
    print("Passages:", conn.scalar(text("select count(*) from passage")))
chat_tag, chat_digest = model.model_tag(model.chat)
print(platform.system(), platform.machine(), platform.python_version())
print(state.revision, state.model, state.dimensions, chat_tag, chat_digest, sep="\n")

Passages: 532
Darwin arm64 3.14.10
2ed99aad2d60f1d6318d6aef5b1c02bfe51716eb
all-minilm:latest@1b226e2802dbb772b5fc32a58f103ca1804ef7501331012de126ab22f67475ef
384
qwen3.5:9b
d63f830f28d515fefea546f213fca7a75e9dd5176ef51a6deabc952d1c99a911


## Paired retrieval experiment

A hit means a top-three passage has the labelled path and overlaps at least one labelled line. This is a location metric. A partial overlap can count even if the passage is insufficient for an answer. An alternative correct implementation path may score as a miss. Timings include client and database work, plus Ollama for the vector method. Cold caches and model loads affect them.

In [2]:
@dataclass(frozen=True)
class RetrievalResult:
    hits: list[Passage]
    elapsed_ms: int
    matched: bool


def matches(part: Passage, case: Question) -> bool:
    lines = case.get("lines")
    if lines is None or case["path"] != part.path:
        return False
    a, b = lines
    return part.start <= b and part.end >= a

runs: dict[str, dict[str, RetrievalResult]] = {}
for case in dataset["questions"]:
    entry: dict[str, RetrievalResult] = {}
    for method in ("text", "vector"):
        start = perf_counter()
        snap, hits = retrieve(db, model, case["question"], method)
        assert snap == state
        entry[method] = RetrievalResult(
            hits=hits,
            elapsed_ms=round(1000 * (perf_counter() - start)),
            matched=any(matches(p, case) for p in hits),
        )
    runs[case["id"]] = entry

for split in ("development", "held-out"):
    cases = [c for c in dataset["questions"] if c["split"] == split and c["path"]]
    for method in ("text", "vector"):
        count = sum(runs[c["id"]][method].matched for c in cases)
        print(f"{split:12} {method:6} Hit@3 {count}/{len(cases)}")
for case in dataset["questions"]:
    r = runs[case["id"]]
    print(case["id"], *(f"{m}: {r[m].matched} ({r[m].elapsed_ms} ms)" for m in ("text","vector")))

development  text   Hit@3 2/4
development  vector Hit@3 2/4
held-out     text   Hit@3 6/6
held-out     vector Hit@3 4/6
inventory-constraint text: False (10 ms) vector: False (80 ms)
rollback-test text: True (7 ms) vector: True (24 ms)
identifier text: True (6 ms) vector: False (22 ms)
normalize text: False (6 ms) vector: True (22 ms)
empty-order text: True (6 ms) vector: False (22 ms)
page text: True (5 ms) vector: True (21 ms)
runtime-user text: True (5 ms) vector: False (22 ms)
movements text: True (7 ms) vector: True (22 ms)
price text: True (6 ms) vector: True (22 ms)
bootstrap text: True (6 ms) vector: True (21 ms)
missing-traffic text: False (5 ms) vector: False (24 ms)
missing-backup text: False (6 ms) vector: False (27 ms)


In [3]:
for case in dataset["questions"]:
    print("\n" + case["question"])
    for method in ("text", "vector"):
        print(method, [(p.path, p.start, p.end) for p in runs[case["id"]][method].hits])


What stops a warehouse from reserving more units than it holds?
text [('README.md', 50, 69), ('README.md', 65, 80), ('src/test/java/dev/bartlomiejmilosz/inventory/InventoryMigrationTests.java', 106, 125)]
vector [('README.md', 65, 80), ('src/main/java/dev/bartlomiejmilosz/inventory/inventory/WarehouseService.java', 16, 35), ('src/main/java/dev/bartlomiejmilosz/inventory/inventory/StockLevel.java', 31, 50)]

Where does a test prove that a later failed write undoes an earlier insert?
text [('src/test/java/dev/bartlomiejmilosz/inventory/InventoryMigrationTests.java', 136, 155), ('src/test/java/dev/bartlomiejmilosz/inventory/EmployeeSecurityTests.java', 271, 290), ('src/test/java/dev/bartlomiejmilosz/inventory/inventory/InventoryCatalogTests.java', 91, 110)]
vector [('src/test/java/dev/bartlomiejmilosz/inventory/InventoryMigrationTests.java', 136, 155), ('src/test/java/dev/bartlomiejmilosz/inventory/InventoryMigrationTests.java', 151, 162), ('src/test/java/dev/bartlomiejmilosz/inventory/o

## Generated answers

One independent request per question. Print Qwen's returned text, exact quote and application-built link. Inspect whether the quote supports the claim as a separate judgement from Hit@3.

In [4]:
answers: dict[str, Answer] = {}
for case in dataset["questions"]:
    start = perf_counter()
    answer = model.answer(case["question"], runs[case["id"]]["vector"].hits)
    answers[case["id"]] = answer
    print("\n", case["id"], "·", round(perf_counter() - start, 2), "s")
    print(case["question"], answer.text, sep="\n")
    if answer.source:
        print(repr(answer.quote), answer.source.link(state.url, state.revision), sep="\n")
missing = [c for c in dataset["questions"] if c["path"] is None]
print("\nMissing-evidence abstentions:", sum(answers[c["id"]].source is None for c in missing), "/", len(missing))


 inventory-constraint · 8.58 s
What stops a warehouse from reserving more units than it holds?
Physical stock cannot fall below reserved stock.
'if (nextOnHand < reserved) {\n      throw new ResourceConflictException("Physical stock cannot fall below reserved stock.");'
https://github.com/bartlomiej-milosz/order-inventory-system/blob/2ed99aad2d60f1d6318d6aef5b1c02bfe51716eb/src/main/java/dev/bartlomiejmilosz/inventory/inventory/StockLevel.java#L31-L50



 rollback-test · 2.55 s
Where does a test prove that a later failed write undoes an earlier insert?
A test proves that a later failed write undoes an earlier insert by checking that the count of rows with the specific code is zero after the transaction fails.
'assertThat(\n            jdbc.queryForObject(\n                "SELECT count(*) FROM inventory.warehouses WHERE code = \'ROLLBACK\'", Integer.class))\n        .isZero();'
https://github.com/bartlomiej-milosz/order-inventory-system/blob/2ed99aad2d60f1d6318d6aef5b1c02bfe51716eb/src/test/java/dev/bartlomiejmilosz/inventory/InventoryMigrationTests.java#L151-L162



 identifier · 1.66 s
What does stock_levels_valid_quantities enforce?
The indexed passages do not contain enough evidence to answer this question.



 normalize · 1.49 s
What happens to whitespace and capitalization in an employee email address?
Whitespace is stripped and capitalization is converted to lowercase.
'return email.strip().toLowerCase(Locale.ROOT);'
https://github.com/bartlomiej-milosz/order-inventory-system/blob/2ed99aad2d60f1d6318d6aef5b1c02bfe51716eb/src/main/java/dev/bartlomiejmilosz/inventory/users/EmailAddress.java#L1-L11



 empty-order · 1.99 s
What happens if someone confirms a draft with no line items?
The indexed passages do not contain enough evidence to answer this question.



 page · 1.96 s
Which pagination fields does PageResponse return?
PageResponse returns items, page, size, totalElements, and totalPages.
'public record PageResponse<T>(\n    List<T> items, int page, int size, long totalElements, int totalPages)'
https://github.com/bartlomiej-milosz/order-inventory-system/blob/2ed99aad2d60f1d6318d6aef5b1c02bfe51716eb/src/main/java/dev/bartlomiejmilosz/inventory/shared/PageResponse.java#L1-L16


... (2 outputs left out)


 price · 1.6 s
What precision and scale does the Product entity use for unit_price?
The Product entity uses precision 12 and scale 2 for unit_price.
'@Column(name = "unit_price", nullable = false, precision = 12, scale = 2)'
https://github.com/bartlomiej-milosz/order-inventory-system/blob/2ed99aad2d60f1d6318d6aef5b1c02bfe51716eb/src/main/java/dev/bartlomiejmilosz/inventory/catalog/Product.java#L1-L20



 bootstrap · 1.5 s
What size limit applies to the bootstrap password file?
256
'if (!Files.isRegularFile(path) || Files.size(path) > 256)'
https://github.com/bartlomiej-milosz/order-inventory-system/blob/2ed99aad2d60f1d6318d6aef5b1c02bfe51716eb/src/main/java/dev/bartlomiejmilosz/inventory/users/AdminBootstrapRunner.java#L31-L50



 missing-traffic · 1.9 s
How many real customer orders did production receive last Tuesday?
The indexed passages do not contain enough evidence to answer this question.



 missing-backup · 2.03 s
What was the UTC timestamp of the last successful production database backup?
The indexed passages do not contain enough evidence to answer this question.

Missing-evidence abstentions: 2 / 2


## Control with manually selected evidence

Take the first answerable vector miss (or the first case if all hit). Ask the same model the same question again with a passage known to contain the beginning of the labelled evidence. This changes only the retrieved context.

In [5]:
case = next((c for c in dataset["questions"] if c["path"] and not runs[c["id"]]["vector"].matched), dataset["questions"][0])
lines = case.get("lines")
if not lines:
    raise ValueError("The manual control requires labelled source lines.")
with db.connect() as conn:
    row = conn.execute(text("""SELECT path,start_line,end_line,body FROM passage
        WHERE path=:path AND start_line <= :line AND end_line >= :line
        ORDER BY end_line DESC LIMIT 1"""), {"path":case["path"], "line":lines[0]}).one()
control = model.answer(case["question"], [Passage(*row)])
print(case["id"], "Retrieved:", answers[case["id"]].text, "Supplied:", control.text, repr(control.quote), sep="\n")
model.close()
db.dispose()

inventory-constraint
Retrieved:
Physical stock cannot fall below reserved stock.
Supplied:
A CHECK constraint ensures reserved units do not exceed on-hand units.
'CONSTRAINT stock_levels_valid_quantities CHECK (\n        on_hand >= 0\n        AND reserved >= 0\n        AND reserved <= on_hand\n    )'


## Interpretation of the saved run

On the six held-out questions with labelled evidence, keyword search hit the expected location 6/6 times and the vector method 4/6. Both scored 2/4 on development cases. Embeddings found the email-normalization method for a natural-language question missed by keyword search. Keywords found the numeric container user, which the vector top three missed. We use keyword search by default for this repository and expose both methods. Choosing a default after observing this set makes a fresh dataset necessary for an independent follow-up estimate.

With vector context, the four held-out answers for pagination, stock movements, price precision and the bootstrap file-size limit cited matching source code. The last answer omitted the unit (bytes). The model abstained on the other two answerable held-out questions and on both questions about undocumented production data. Two abstentions are a small control, not a general hallucination rate. A valid substring alone cannot establish that a generated claim follows from it.

Both retrieval methods missed the labelled SQL for the warehouse question. The vector result cited a related Java guard about lowering physical stock. With the labelled SQL supplied directly, Qwen quoted the actual CHECK constraint. This explains why a plausible answer and a hit on the expected evidence are separate observations.

These counts describe one snapshot, one embedding model, hand-labelled questions and one run. Short passages may divide a method, and the embedding model may truncate long input. A second label for an equally valid passage or a different model can change the scores. Retrieval timings for this run were 5–10 ms for keyword search and 21–80 ms for vector search; generation typically took substantially longer. Use the per-question output when reproducing the run on other hardware.